# Real-ESRGAN Batch Upscaler dengan Tombol GUI (Google Colab GPU)

Upscale kumpulan gambar secara massal dengan GPU T4 di Google Colab.  
- **Pilihan Skala**: `2x` (ukuran pas, tajam) atau `4x` (detail maksimal hingga puluhan megapixel).
- **Format Output**: JPG Kualitas Tinggi (Quality 96, Subsampling 4:4:4, warna padat dan tidak drop).
- **Fitur Tombol GUI**: Pilih Skala, Mulai Upscale, Unduh ZIP, dan Bersihkan Folder (agar batch tidak tercampur).
- **Engine**: Pure PyTorch bawaan Colab (Tanpa basicsr, anti-error).

---  
**Pastikan Runtime GPU Aktif**:  
Menu `Runtime` -> `Change runtime type` -> pilih `T4 GPU` -> `Save`.

In [ ]:
# 1. Periksa GPU (Harus muncul Tesla T4)
!nvidia-smi

In [ ]:
# 2. Siapkan folder dan unduh model RealESRGAN_x4plus
import os
!mkdir -p weights inputs outputs

print("Memeriksa bobot model...")
!wget -c -q --show-progress "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth" -P weights/
print("Model siap digunakan!")

In [ ]:
# 3. Siapkan script runner mandiri
%%writefile upscale_batch.py
import argparse, glob, math, os, shutil, sys
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F

class ResidualDenseBlock(nn.Module):
    def __init__(self, num_feat=64, num_grow_ch=32):
        super().__init__()
        self.conv1 = nn.Conv2d(num_feat, num_grow_ch, 3, 1, 1)
        self.conv2 = nn.Conv2d(num_feat + num_grow_ch, num_grow_ch, 3, 1, 1)
        self.conv3 = nn.Conv2d(num_feat + 2 * num_grow_ch, num_grow_ch, 3, 1, 1)
        self.conv4 = nn.Conv2d(num_feat + 3 * num_grow_ch, num_grow_ch, 3, 1, 1)
        self.conv5 = nn.Conv2d(num_feat + 4 * num_grow_ch, num_feat, 3, 1, 1)
        self.lrelu = nn.LeakyReLU(negative_slope=0.2, inplace=True)
    def forward(self, x):
        x1 = self.lrelu(self.conv1(x))
        x2 = self.lrelu(self.conv2(torch.cat((x, x1), 1)))
        x3 = self.lrelu(self.conv3(torch.cat((x, x1, x2), 1)))
        x4 = self.lrelu(self.conv4(torch.cat((x, x1, x2, x3), 1)))
        x5 = self.conv5(torch.cat((x, x1, x2, x3, x4), 1))
        return x5 * 0.2 + x

class RRDB(nn.Module):
    def __init__(self, num_feat=64, num_grow_ch=32):
        super().__init__()
        self.rdb1 = ResidualDenseBlock(num_feat, num_grow_ch)
        self.rdb2 = ResidualDenseBlock(num_feat, num_grow_ch)
        self.rdb3 = ResidualDenseBlock(num_feat, num_grow_ch)
    def forward(self, x):
        out = self.rdb1(x)
        out = self.rdb2(out)
        out = self.rdb3(out)
        return out * 0.2 + x

class RRDBNet(nn.Module):
    def __init__(self, num_in_ch=3, num_out_ch=3, scale=4, num_feat=64, num_block=23, num_grow_ch=32):
        super().__init__()
        self.scale = scale
        self.conv_first = nn.Conv2d(num_in_ch, num_feat, 3, 1, 1)
        self.body = nn.Sequential(*[RRDB(num_feat, num_grow_ch) for _ in range(num_block)])
        self.conv_body = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.conv_up1 = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.conv_up2 = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.conv_hr = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.conv_last = nn.Conv2d(num_feat, num_out_ch, 3, 1, 1)
        self.lrelu = nn.LeakyReLU(negative_slope=0.2, inplace=True)
    def forward(self, x):
        feat = self.conv_first(x)
        body_feat = self.conv_body(self.body(feat))
        feat = feat + body_feat
        feat = self.lrelu(self.conv_up1(F.interpolate(feat, scale_factor=2, mode="nearest")))
        feat = self.lrelu(self.conv_up2(F.interpolate(feat, scale_factor=2, mode="nearest")))
        out = self.conv_last(self.lrelu(self.conv_hr(feat)))
        return out

def load_model(model_path, device, fp16=True):
    model = RRDBNet(num_in_ch=3, num_out_ch=3, scale=4, num_feat=64, num_block=23, num_grow_ch=32)
    try:
        checkpoint = torch.load(model_path, map_location=device, weights_only=False)
    except TypeError:
        checkpoint = torch.load(model_path, map_location=device)
    state_dict = checkpoint.get("params_ema", checkpoint.get("params", checkpoint))
    model.load_state_dict(state_dict, strict=True)
    model.eval().to(device)
    return model.half() if (fp16 and device.type == "cuda") else model

def process_tiles(img_tensor, model, tile=512, tile_pad=10, scale=4):
    batch, channel, height, width = img_tensor.shape
    if tile == 0 or (height <= tile and width <= tile):
        with torch.no_grad():
            return model(img_tensor)
    out_tensor = torch.zeros((batch, channel, height * scale, width * scale), dtype=img_tensor.dtype, device=img_tensor.device)
    tiles_x = math.ceil(width / tile)
    tiles_y = math.ceil(height / tile)
    for y in range(tiles_y):
        for x in range(tiles_x):
            in_x = x * tile
            in_y = y * tile
            in_x_end = min(in_x + tile, width)
            in_y_end = min(in_y + tile, height)
            in_x_pad = max(in_x - tile_pad, 0)
            in_x_end_pad = min(in_x_end + tile_pad, width)
            in_y_pad = max(in_y - tile_pad, 0)
            in_y_end_pad = min(in_y_end + tile_pad, height)
            tile_in = img_tensor[:, :, in_y_pad:in_y_end_pad, in_x_pad:in_x_end_pad]
            with torch.no_grad():
                tile_out = model(tile_in)
            out_x = in_x * scale
            out_x_end = in_x_end * scale
            out_y = in_y * scale
            out_y_end = in_y_end * scale
            t_out_x = (in_x - in_x_pad) * scale
            t_out_x_end = t_out_x + (in_x_end - in_x) * scale
            t_out_y = (in_y - in_y_pad) * scale
            t_out_y_end = t_out_y + (in_y_end - in_y) * scale
            out_tensor[:, :, out_y:out_y_end, out_x:out_x_end] = tile_out[:, :, t_out_y:t_out_y_end, t_out_x:t_out_x_end]
    return out_tensor

def upscale_image(pil_img, model, device, tile=512, fp16=True, outscale=4.0):
    orig_w, orig_h = pil_img.size
    img = pil_img.convert("RGB")
    np_img = np.array(img, dtype=np.float32) / 255.0
    tensor = torch.from_numpy(np_img).permute(2, 0, 1).unsqueeze(0).to(device)
    if fp16 and device.type == "cuda":
        tensor = tensor.half()
    out_tensor = process_tiles(tensor, model, tile=tile, tile_pad=10, scale=4)
    out_tensor = out_tensor.squeeze(0).float().clamp(0.0, 1.0)
    out_np = (out_tensor.permute(1, 2, 0).cpu().numpy() * 255.0).round().astype(np.uint8)
    res_img = Image.fromarray(out_np)
    if outscale != 4.0:
        res_img = res_img.resize((int(round(orig_w * outscale)), int(round(orig_h * outscale))), Image.Resampling.LANCZOS)
    return res_img

def clean_folders(input_dir="inputs", output_dir="outputs"):
    for folder in [input_dir, output_dir]:
        if os.path.exists(folder):
            for f in os.listdir(folder):
                p = os.path.join(folder, f)
                if os.path.isfile(p): os.remove(p)
                elif os.path.isdir(p): shutil.rmtree(p)
    if os.path.exists("hasil_upscale.zip"): os.remove("hasil_upscale.zip")
    print("Folder inputs, outputs, dan file zip dibersihkan.")

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("-i", "--input", type=str, default="inputs")
    parser.add_argument("-o", "--output", type=str, default="outputs")
    parser.add_argument("-m", "--model_path", type=str, default="weights/RealESRGAN_x4plus.pth")
    parser.add_argument("-s", "--scale", type=float, default=4.0, choices=[2.0, 4.0])
    parser.add_argument("-t", "--tile", type=int, default=512)
    parser.add_argument("-q", "--quality", type=int, default=96)
    parser.add_argument("--fp16", action="store_true", default=True)
    parser.add_argument("--clean", action="store_true")
    args = parser.parse_args()
    if args.clean:
        clean_folders(args.input, args.output)
        return
    os.makedirs(args.output, exist_ok=True)
    os.makedirs(args.input, exist_ok=True)
    exts = ("*.png", "*.jpg", "*.jpeg", "*.webp", "*.bmp")
    files = []
    for ext in exts:
        files.extend(glob.glob(os.path.join(args.input, ext)))
        files.extend(glob.glob(os.path.join(args.input, ext.upper())))
    if not files and os.path.exists("/content"):
        for ext in valid_exts:
            for f in glob.glob(f"/content/{ext}") + glob.glob(f"/content/{ext.upper()}"):
                dest = os.path.join(args.input, os.path.basename(f))
                os.rename(f, dest)
                files.append(dest)
    files = sorted(list(set(files)))
    if not files:
        print(f"ERROR: Tidak ditemukan gambar di '{args.input}' atau '/content/'. Upload gambar terlebih dahulu!")
        sys.exit(1)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device} | Memuat model {args.model_path}...")
    model = load_model(args.model_path, device, fp16=args.fp16)
    print(f"Ditemukan {len(files)} gambar | Skala: {args.scale}x | Kualitas JPG: {args.quality}")
    success_count = 0
    for i, path in enumerate(files, 1):
        name = os.path.splitext(os.path.basename(path))[0]
        out_path = os.path.join(args.output, f"{name}_upscaled_{int(args.scale)}x.jpg")
        print(f"[{i}/{len(files)}] {os.path.basename(path)} -> ", end="", flush=True)
        try:
            with Image.open(path) as src_img:
                out_img = upscale_image(src_img, model, device, tile=args.tile, fp16=args.fp16, outscale=args.scale)
                out_img.save(out_path, format="JPEG", quality=args.quality, subsampling=0, optimize=True)
                w, h = out_img.size
                print(f"SELESAI ({w}x{h} px)")
                success_count += 1
        except Exception as e:
            print(f"GAGAL: {e}")
    print(f"\nRingkasan: {success_count}/{len(files)} gambar berhasil di-upscale.")
    if success_count == 0:
        sys.exit(1)

if __name__ == "__main__":
    main()


In [ ]:
# 4. DASHBOARD INTERAKTIF (TOMBOL GUI LENGKAP)
import os, glob, shutil
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from google.colab import files

# Komponen Kontrol
scale_choice = widgets.ToggleButtons(
    options=[('2x (Pas & Tajam)', 2.0), ('4x (Maksimal Detail)', 4.0)],
    value=2.0,
    description='Perbesaran:',
    button_style='info'
)

quality_slider = widgets.IntSlider(
    value=96, min=85, max=100, step=1,
    description='Kualitas JPG:',
    continuous_update=False
)

btn_upload = widgets.Button(
    description='📁 Upload Gambar Bahan',
    button_style='',
    tooltip='Upload file gambar ke folder inputs/',
    icon='upload'
)

btn_run = widgets.Button(
    description='🚀 Mulai Upscale',
    button_style='primary',
    tooltip='Jalankan proses upscale untuk semua gambar di inputs/',
    icon='play'
)

btn_download = widgets.Button(
    description='📦 Unduh Hasil (ZIP)',
    button_style='success',
    tooltip='Kompres folder outputs/ dan download ke laptop',
    icon='download'
)

btn_clean = widgets.Button(
    description='🧹 Bersihkan Folder (Reset)',
    button_style='danger',
    tooltip='Hapus semua file inputs, outputs, dan zip agar tidak tercampur batch berikutnya',
    icon='trash'
)

log_output = widgets.Output()

def on_upload_clicked(b):
    with log_output:
        clear_output()
        os.makedirs('inputs', exist_ok=True)
        print("Pilih gambar dari komputer Anda:")
        up = files.upload()
        for fn in up.keys():
            os.rename(fn, os.path.join('inputs', fn))
        in_count = len(glob.glob('inputs/*'))
        print(f"✅ Total {in_count} gambar saat ini ada di folder inputs/.")

def on_run_clicked(b):
    with log_output:
        clear_output()
        s = scale_choice.value
        q = quality_slider.value
        print(f"Memulai Upscale dengan Skala {s}x, Kualitas {q}...")
        cmd = f"python upscale_batch.py --input inputs --output outputs --scale {s} --quality {q} --tile 512"
        os.system(cmd)

def on_download_clicked(b):
    with log_output:
        clear_output()
        out_files = glob.glob('outputs/*_upscaled_*.jpg') + glob.glob('outputs/*_upscaled.jpg')
        if not out_files:
            print("❌ Folder outputs/ KOSONG! Tekan tombol 'Mulai Upscale' terlebih dahulu.")
            return
        print(f"Mengompres {len(out_files)} gambar ke hasil_upscale.zip...")
        shutil.make_archive('hasil_upscale', 'zip', 'outputs')
        size_mb = os.path.getsize('hasil_upscale.zip') / (1024 * 1024)
        print(f"Ukuran file: {size_mb:.2f} MB. Memulai unduhan...")
        files.download('hasil_upscale.zip')

def on_clean_clicked(b):
    with log_output:
        clear_output()
        os.system("python upscale_batch.py --clean")
        print("✅ Folder 'inputs/' dan 'outputs/' BERSIH! Siap untuk batch baru.")

btn_upload.on_click(on_upload_clicked)
btn_run.on_click(on_run_clicked)
btn_download.on_click(on_download_clicked)
btn_clean.on_click(on_clean_clicked)

panel = widgets.VBox([
    widgets.HTML("<h3>🛠️ Kontrol Image Upscaler</h3>"),
    scale_choice,
    quality_slider,
    widgets.HBox([btn_upload, btn_run, btn_download, btn_clean]),
    widgets.HTML("<h4>📋 Log Status:</h4>"),
    log_output
])

display(panel)